# Evaluating the retriever with deepeval

Stage 1 of the eval suite: the retriever on its own, without the answering model. Each question in
`evals/retriever_goldens.json` goes straight to the agent's own `search_document` tool, so it gets exactly the chunks
the agent would: the top `RETRIEVED_CHUNK_COUNT` (3), with scores below `MIN_RELEVANCE_SCORE` (0.3) dropped.

deepeval scores what comes back with two metrics, both judged by DeepSeek Flash:

| Metric | What it asks | What it reads |
|---|---|---|
| Contextual Recall | Can every sentence of the reference answer be found in the retrieved chunks? | The reference answer and the chunks |
| Contextual Precision | Are the chunks that help ranked above the ones that do not? | The question, the reference answer and the chunks, in rank order |

Both metrics require an `actual_output` but do not score it, so every test case gets `"generator not evaluated"`.

The golden set has 15 questions written from the PDF: 5 on text, 5 on tables, 3 on charts, and 2 whose answer spans
two pages. A run makes 15 Jina searches (about 15 tokens each) and about 60 DeepSeek Flash calls (a verdict and a
reason per metric and question).

In [1]:
import json
import os
import sys
import time
from pathlib import Path

# Set before deepeval and the project's config are imported, since both read them then
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"  # deepeval sends no usage data
os.environ["LANGSMITH_TRACING"] = "false"  # keeps the eval's searches out of the LangSmith project

PROJECT_ROOT = Path.cwd().parent  # this notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

from deepeval import evaluate
from deepeval.evaluate.configs import CacheConfig, DisplayConfig, ErrorConfig
from deepeval.metrics import ContextualPrecisionMetric, ContextualRecallMetric
from deepeval.models import DeepSeekModel
from deepeval.test_case import LLMTestCase
from IPython.display import Markdown, display

from multimodal_rag import config
from multimodal_rag.generate import search_document  # importing it opens the vector store

GOLDENS_PATH = PROJECT_ROOT / "evals" / "retriever_goldens.json"

# Judge model, on DeepSeek's own API. Its key is DEEPSEEK_API_KEY in .env.
JUDGE_MODEL = "deepseek-flash"  # DeepSeek-V4.1-Flash
# deepeval has no price for deepseek-flash, so it gets DeepSeek's peak list price, in USD per token.
# Off-peak hours cost half and cached input much less, so the reported cost is an upper bound.
INPUT_PRICE = 0.30 / 1_000_000
OUTPUT_PRICE = 1.20 / 1_000_000

print("Retrieved chunks per search:", config.RETRIEVED_CHUNK_COUNT, "| minimum relevance score:", config.MIN_RELEVANCE_SCORE)

Retrieved chunks per search: 3 | minimum relevance score: 0.3


## 1. Retrieve

Each question is sent to `search_document` as a tool call, which returns the chunks as the tool message's artifact,
the same list the agent receives. Pages are PDF pages, and the score is the vector store's relevance score.

In [2]:
goldens = json.loads(GOLDENS_PATH.read_text(encoding="utf-8"))

test_cases = []
retrieved = []  # for each golden, the (page, score) of each chunk, in rank order
for number, golden in enumerate(goldens, start=1):
    tool_call = {"name": "search_document", "args": {"query": golden["input"]}, "id": f"golden-{number}", "type": "tool_call"}
    chunks = search_document.invoke(tool_call).artifact
    retrieved.append([(chunk["page_number"], round(chunk["score"], 2)) for chunk in chunks])
    test_cases.append(
        LLMTestCase(
            name=f"golden-{number}",
            input=golden["input"],
            actual_output="generator not evaluated",  # required by both metrics, scored by neither
            expected_output=golden["expected_output"],
            retrieval_context=[chunk["text"] for chunk in chunks],
        )
    )
    metadata = golden["additional_metadata"]
    print(f"{number:2}. {metadata['content']:10}  expected pages {metadata['pdf_pages']}  retrieved (page, score) {retrieved[-1]}")

 1. text        expected pages [149]  retrieved (page, score) [(149, 0.62), (24, 0.49), (143, 0.47)]


 2. text        expected pages [121]  retrieved (page, score) [(150, 0.59), (24, 0.54), (67, 0.53)]


 3. text        expected pages [125]  retrieved (page, score) [(23, 0.71), (125, 0.63), (129, 0.62)]


 4. text        expected pages [99]  retrieved (page, score) [(124, 0.49), (93, 0.48), (82, 0.46)]


 5. text        expected pages [20]  retrieved (page, score) [(25, 0.74), (20, 0.69), (29, 0.66)]


...

11. chart       expected pages [7]  retrieved (page, score) [(7, 0.66), (7, 0.63), (17, 0.47)]


12. chart       expected pages [24]  retrieved (page, score) [(24, 0.73), (24, 0.68), (142, 0.57)]


13. chart       expected pages [57]  retrieved (page, score) [(57, 0.72), (65, 0.7), (62, 0.67)]


14. multi_page  expected pages [44, 45]  retrieved (page, score) [(50, 0.68), (45, 0.6), (45, 0.6)]


15. multi_page  expected pages [153, 154]  retrieved (page, score) [(141, 0.62), (143, 0.61), (23, 0.57)]


## 2. Score

deepeval runs both metrics on every test case, several at a time. Each metric also writes the reason for its score.

In [3]:
judge = DeepSeekModel(
    model=JUDGE_MODEL,
    api_key=os.environ["DEEPSEEK_API_KEY"],
    cost_per_input_token=INPUT_PRICE,
    cost_per_output_token=OUTPUT_PRICE,
)
metrics = [ContextualPrecisionMetric(model=judge), ContextualRecallMetric(model=judge)]

start = time.perf_counter()
result = evaluate(
    test_cases=test_cases,
    metrics=metrics,
    display_config=DisplayConfig(show_indicator=False, print_results=False, inspect_after_run=False),
    cache_config=CacheConfig(write_cache=False),
    error_config=ErrorConfig(ignore_errors=True),  # a failed judge call shows as an error below instead of stopping the run
)
seconds = time.perf_counter() - start

scores = {test_result.name: {data.name: data for data in test_result.metrics_data} for test_result in result.test_results}
all_metric_data = [data for by_name in scores.values() for data in by_name.values()]
cost = sum(data.evaluation_cost or 0 for data in all_metric_data)
input_tokens = sum(data.input_tokens or 0 for data in all_metric_data)
output_tokens = sum(data.output_tokens or 0 for data in all_metric_data)
print(f"Scored {len(result.test_results)} test cases in {seconds:.0f} seconds")
print(f"Judge tokens: {input_tokens:,} input, {output_tokens:,} output; cost at most ${cost:.4f}")

⚠ WARNING: No hyperparameters logged.
» ]8;id=5104330;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 11.5s | token cost: 0.0590541 USD)
» Test Results (15 total tests):
   » Pass Rate: 60.0% | Passed: 9 | Failed: 6

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Scored 15 test cases in 12 seconds
Judge tokens: 82,995 input, 28,463 output; cost at most $0.0591


## 3. Results

Scores run from 0 to 1. Retrieved pages are in rank order, with an expected page in bold.

In [4]:
def score_text(data):
    return "error" if data.score is None else f"{data.score:.2f}"


rows = [
    "| # | Content | Question | Expected pages | Retrieved pages (score) | Precision | Recall |",
    "|---|---|---|---|---|---|---|",
]
by_content = {}  # content type -> list of (precision, recall)
for number, golden in enumerate(goldens, start=1):
    metadata = golden["additional_metadata"]
    expected = metadata["pdf_pages"]
    pages = ", ".join(f"**{page}** ({score})" if page in expected else f"{page} ({score})" for page, score in retrieved[number - 1])
    precision = scores[f"golden-{number}"]["Contextual Precision"]
    recall = scores[f"golden-{number}"]["Contextual Recall"]
    rows.append(
        f"| {number} | {metadata['content']} | {golden['input']} | {', '.join(map(str, expected))} | {pages or 'none'} "
        f"| {score_text(precision)} | {score_text(recall)} |"
    )
    by_content.setdefault(metadata["content"], []).append((precision.score, recall.score))
display(Markdown("\n".join(rows)))


def mean(values):
    values = [value for value in values if value is not None]
    return sum(values) / len(values) if values else float("nan")


summary = ["| Content | Questions | Mean precision | Mean recall |", "|---|---|---|---|"]
for content, pairs in by_content.items():
    summary.append(f"| {content} | {len(pairs)} | {mean(p for p, _ in pairs):.2f} | {mean(r for _, r in pairs):.2f} |")
all_pairs = [pair for pairs in by_content.values() for pair in pairs]
summary.append(f"| **all** | {len(all_pairs)} | **{mean(p for p, _ in all_pairs):.2f}** | **{mean(r for _, r in all_pairs):.2f}** |")
display(Markdown("\n".join(summary)))

| # | Content | Question | Expected pages | Retrieved pages (score) | Precision | Recall |
|---|---|---|---|---|---|---|
| 1 | text | What did NVIDIA get from its December 2025 agreement with Groq, and how much did it pay? | 149 | **149** (0.62), 24 (0.49), 143 (0.47) | 1.00 | 1.00 |
| 2 | text | How much room does NVIDIA still have to buy back its own shares? | 121 | 150 (0.59), 24 (0.54), 67 (0.53) | 0.50 | 0.00 |
| 3 | text | Why did NVIDIA's gross margin go down in fiscal 2026? | 125 | 23 (0.71), **125** (0.63), 129 (0.62) | 1.00 | 1.00 |
| 4 | text | How big is NVIDIA's workforce, and how many people work in research? | 99 | 124 (0.49), 93 (0.48), 82 (0.46) | 0.33 | 0.00 |
| 5 | text | When and where is the 2026 annual shareholder meeting, and who can vote at it? | 20 | 25 (0.74), **20** (0.69), 29 (0.66) | 1.00 | 1.00 |
| 6 | table | What did NVIDIA pay its auditor PwC in fiscal 2026, and how much of that was for the audit itself? | 78 | 79 (0.68), **78** (0.67), 137 (0.57) | 0.50 | 1.00 |
| 7 | table | How much of NVIDIA's Data Center revenue came from compute and how much from networking? | 167 | 128 (0.66), 93 (0.65), 23 (0.65) | 0.00 | 0.00 |
| 8 | table | How much revenue came from customers based in China, and how did that change from the year before? | 166 | **166** (0.49), 129 (0.45), 167 (0.41) | 1.00 | 1.00 |
| 9 | table | How much were NVIDIA's non-employee directors paid for fiscal 2026? | 52 | 74 (0.71), 57 (0.71), 61 (0.7) | 0.00 | 0.00 |
| 10 | table | Who are NVIDIA's largest shareholders? | 54 | 67 (0.58), 174 (0.56), 142 (0.56) | 0.00 | 0.00 |
| 11 | chart | Which organizations publish the most open-source AI repositories, according to the report? | 7 | **7** (0.66), **7** (0.63), 17 (0.47) | 1.00 | 1.00 |
| 12 | chart | How much cash has NVIDIA returned to shareholders over the past one, three and five years? | 24 | **24** (0.73), **24** (0.68), 142 (0.57) | 1.00 | 1.00 |
| 13 | chart | How is the CEO's target pay for fiscal 2026 split between salary, cash and stock awards? | 57 | **57** (0.72), 65 (0.7), 62 (0.67) | 1.00 | 0.67 |
| 14 | multi_page | Who chairs each of the Board's committees, and how often did each committee meet in fiscal 2026? | 44, 45 | 50 (0.68), **45** (0.6), **45** (0.6) | 0.58 | 0.50 |
| 15 | multi_page | How much did NVIDIA hold in cash equivalents and marketable securities at the end of fiscal 2026, compared with a year earlier? | 153, 154 | 141 (0.62), 143 (0.61), 23 (0.57) | 1.00 | 0.00 |

| Content | Questions | Mean precision | Mean recall |
|---|---|---|---|
| text | 5 | 0.77 | 0.60 |
| table | 5 | 0.30 | 0.40 |
| chart | 3 | 1.00 | 0.89 |
| multi_page | 2 | 0.79 | 0.25 |
| **all** | 15 | **0.66** | **0.54** |

## 4. Reasons

The judge's reason for every score below 1, and any error.

In [5]:
for number, golden in enumerate(goldens, start=1):
    for name, data in scores[f"golden-{number}"].items():
        if data.score is None or data.score < 1:
            print(f"{number:2}. {name} {score_text(data)}: {data.error or data.reason}\n")

 2. Contextual Precision 0.50: The score is 0.50 because the only relevant node is ranked second in the retrieval contexts; the first node is irrelevant since it discusses the '2012 Employee Stock Purchase Plan' and shares 'reserved for future issuance under the 2012 Plan,' and the third node is irrelevant because it covers 'Stock Ownership Guidelines'—both should be ranked lower than the relevant second node. It is not higher because the relevant second node only supports the '$40.4 billion' repurchase figure and does not address remaining buyback authorization, while an irrelevant node appears ahead of it.

 2. Contextual Recall 0.00: The score is 0.00 because no expected-output sentence is fully attributable: sentence 1's $58.5 billion repurchase authorization is absent from node 1 in retrieval context, which only ties January 25, 2026 to 2.2 billion shares reserved; sentence 2's August 26, 2025 Board approval of $60.0 billion without expiration is unsupported by any node; and sente